In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra

import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ieee-fraud-detection/sample_submission.csv
/kaggle/input/ieee-fraud-detection/test_identity.csv
/kaggle/input/ieee-fraud-detection/train_identity.csv
/kaggle/input/ieee-fraud-detection/test_transaction.csv
/kaggle/input/ieee-fraud-detection/train_transaction.csv


In [3]:
!pip install mlflow dagshub --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 32.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 69.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 692.3/692.3 kB 18.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed.

In [4]:
!pip install mlflow dagshub --quiet
import mlflow
from dagshub import dagshub_logger
import os

# Set tracking URI manually
mlflow.set_tracking_uri("https://dagshub.com/ekvirika/FraudDerection.mlflow")

# Use your DagsHub credentials
os.environ["MLFLOW_TRACKING_USERNAME"] = "ekvirika"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "3f601f2c2c7a6bca448ebc69f4f5d4b49daffd8f"

# Optional: set registry if you're using model registry
mlflow.set_registry_uri("https://dagshub.com/ekvirika/FraudDerection.mlflow")

In [4]:
import dagshub
dagshub.init(repo_owner='ekvirika', repo_name='FraudDerection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=91bd5a4e-07da-4c21-a1a6-fcdf1ac8e555&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=803c804db1f6b0ca4b22327542c3e5789fb19b0aedf755a0a7733be81b30ae96




Accessing as ekvirika

Initialized MLflow to track repo "ekvirika/FraudDerection"

Repository ekvirika/FraudDerection initialized!

In [4]:
import mlflow
mlflow.set_experiment("RandomForest_Training")

<Experiment: artifact_location='mlflow-artifacts:/32ab6f483e4841628736b89cb8eb10c0', creation_time=1744702177290, experiment_id='3', last_update_time=1744702177290, lifecycle_stage='active', name='RandomForest_Training', tags={}>

Load file from mlflow experiments with user_id

In [6]:
run_id = '78bbe86508204c1388aaa3ae689133df'
artifact_path = "user_id_datasets/card1_addr1/dataset_card1_addr1.parquet"
local_path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=artifact_path)

df = pd.read_parquet(local_path)

In [5]:
df = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_transaction.csv')

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.sklearn
import xgboost as xgb
from category_encoders import WOEEncoder
import warnings
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from imblearn.pipeline import Pipeline as ImbPipeline  # ✅ Corrected import
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import shap
from sklearn.ensemble import RandomForestClassifier
import mlflow.data
from mlflow.data.pandas_dataset import PandasDataset
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns


warnings.filterwarnings('ignore')

# Assume df is already loaded and merged
mlflow.set_experiment('RandomForest_Training')

with mlflow.start_run(run_name="RandomForest_GridSearch"):

    y = df['isFraud']
    X = df.drop(columns=['isFraud', 'TransactionID'])

    # Drop high-missing columns
    missing_ratio = X.isnull().mean()
    cols_to_drop = missing_ratio[missing_ratio > 0.9].index.tolist()
    X.drop(columns=cols_to_drop, inplace=True)
    mlflow.log_param("dropped_cols_90pct_na", len(cols_to_drop))

    # Identify types
    cat_cols = X.select_dtypes(include='object').columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    mlflow.log_param("categorical_features", len(cat_cols))
    mlflow.log_param("numerical_features", len(num_cols))

    # Train-test split by user
    user_list = df["user_id"].unique()
    train_users, valid_users = train_test_split(user_list, test_size=0.2, random_state=42)

    train_mask = df["user_id"].isin(train_users)
    valid_mask = df["user_id"].isin(valid_users)

    X_train = df[train_mask].drop(columns=["isFraud"])
    y_train = df[train_mask]["isFraud"]
    X_valid = df[valid_mask].drop(columns=["isFraud"])
    y_valid = df[valid_mask]["isFraud"]

    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", WOEEncoder())
    ])
    preprocessor = ColumnTransformer([
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ])

    full_pipeline = ImbPipeline([
        ("preprocessor", preprocessor),
        ("sampler", RandomUnderSampler(random_state=42)),  # 👈 Replaced SMOTE
        ("clf", RandomForestClassifier(random_state=42))
    ])

    
    param_grid = {
        "clf__n_estimators": [120, 150, 200],
        "clf__max_depth": [11, 12, 15],
        "clf__class_weight": ["balanced"]
    }


    search = GridSearchCV(
        full_pipeline,
        param_grid,
        scoring='roc_auc',
        cv=2,  # ⏱️ faster
        verbose=1,
        n_jobs=-1  # ⏱️ parallel processing
    )

    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    best_params = search.best_params_
    
    # Log best params
    for param, val in best_params.items():
        mlflow.log_param(param, val)
    
    # Predict and evaluate
    y_pred = best_model.predict(X_valid)
    y_pred_proba = best_model.predict_proba(X_valid)[:, 1]
    
    # Compute metrics
    auc = roc_auc_score(y_valid, y_pred_proba)
    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    cm = confusion_matrix(y_valid, y_pred)
    
    # Log metrics
    mlflow.log_metric("val_auc", auc)
    mlflow.log_metric("val_accuracy", accuracy)
    mlflow.log_metric("val_precision", precision)
    mlflow.log_metric("val_recall", recall)
    mlflow.log_metric("val_f1", f1)
    
    # Log confusion matrix as image
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    
    # Save the model
    mlflow.sklearn.log_model(best_model, "RandomForest_pipeline")
    
    print(f"Best AUC: {auc:.4f}")
    print("Best Parameters:")
    print(best_params)


2025/04/20 14:07:23 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest_Training' does not exist. Creating a new experiment.


NameError: name 'df' is not defined

In [6]:
!pip install imbalanced-learn==0.11.0 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.6/235.6 kB 4.8 MB/s eta 0:00:0000:01


In [7]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import BaseEstimator, TransformerMixin

# --- Time Features including weekdays and months
class TimeFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['Transaction_datetime'] = pd.to_datetime(X['TransactionDT'], unit='s', origin='2017-12-01')
        X['Transaction_hour'] = X['Transaction_datetime'].dt.hour
        X['Transaction_day'] = X['Transaction_datetime'].dt.day
        X['Transaction_weekday'] = X['Transaction_datetime'].dt.weekday
        X['Transaction_month'] = X['Transaction_datetime'].dt.month
        X['Transaction_year'] = X['Transaction_datetime'].dt.year
        
        # Drop the datetime column as it's not needed after extraction
        X = X.drop('Transaction_datetime', axis=1)
        return X

class UserIDCreator(BaseEstimator, TransformerMixin):
    def __init__(self, version=1):
        self.version = version

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.version == 1:
            # Convert to string and fill NAs with placeholder
            card1 = X['card1'].fillna(-999).astype(str)
            day = X['Transaction_day'].fillna(-1).astype(str)
            hour = X['Transaction_hour'].fillna(-1).astype(str)
            X['user_id'] = card1 + '_' + day + '_' + hour
        else:
            # Handle potential NAs in P_emaildomain
            card1 = X['card1'].fillna(-999).astype(str)
            email = X['P_emaildomain'].fillna('unknown').astype(str)
            day = X['Transaction_day'].fillna(-1).astype(str)
            X['user_id'] = card1 + '_' + email + '_' + day
        return X

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in self.columns:
            # Handle NaN values
            self.freq_maps[col] = X[col].fillna('NaN').value_counts() / len(X)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            # Apply encoding and handle unseen values
            X[col + '_freq_enc'] = X[col].fillna('NaN').map(self.freq_maps[col]).fillna(0)
        return X

class WOEEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns, target_col, min_samples=100):
        self.columns = columns
        self.target_col = target_col
        self.woe_maps = {}
        self.min_samples = min_samples
        self.global_mean = None

    def fit(self, X, y):
        # Store global mean for rare categories
        self.global_mean = y.mean()
        
        for col in self.columns:
            # Create DataFrame with target
            df = pd.DataFrame({col: X[col], self.target_col: y})
            
            # Group and calculate WOE
            groups = df.groupby(col)[self.target_col]
            counts = groups.count()
            
            # Filter out rare categories
            valid_cats = counts[counts >= self.min_samples].index
            
            event = groups.sum()
            non_event = counts - event
            
            # Calculate WOE with smoothing to avoid inf values
            woe = np.log(((event + 0.5) / (event.sum() + 0.5)) / 
                        ((non_event + 0.5) / (non_event.sum() + 0.5)))
            
            # Store only for valid categories
            self.woe_maps[col] = woe[valid_cats]
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            # Apply mapping with fallback for unseen categories
            X[col + '_woe_enc'] = X[col].map(self.woe_maps[col]).fillna(0)
        return X

class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns, target_col, min_samples=100, smoothing=10):
        self.columns = columns
        self.target_col = target_col
        self.target_means = {}
        self.min_samples = min_samples
        self.smoothing = smoothing
        self.global_mean = None

    def fit(self, X, y):
        self.global_mean = y.mean()
        
        for col in self.columns:
            # Create DataFrame with target
            df = pd.DataFrame({col: X[col], self.target_col: y})
            
            # Calculate means and counts
            agg = df.groupby(col)[self.target_col].agg(['mean', 'count'])
            
            # Apply smoothing: (count * mean + smoothing * global_mean) / (count + smoothing)
            smoothed_means = (agg['count'] * agg['mean'] + self.smoothing * self.global_mean) / (agg['count'] + self.smoothing)
            
            # Store only categories with enough samples
            valid_cats = agg[agg['count'] >= self.min_samples].index
            self.target_means[col] = smoothed_means[valid_cats]
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            # Apply mapping with fallback to global mean for unseen categories
            X[col + '_target_enc'] = X[col].map(self.target_means[col]).fillna(self.global_mean)
        return X

# --- Corrected Feature Pipeline
def create_preprocessing_pipeline():
    # Identify column types
    numeric_features = ['TransactionAmt', 'card1', 'card2', 'card3', 'card5',
                         'addr1', 'addr2', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 
                         'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14',
                         'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8',
                         'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15']
    
    categorical_features = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 
                           'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5',
                           'M6', 'M7', 'M8', 'M9', 'DeviceType', 'DeviceInfo']
    
    # Create preprocessing steps
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median'))
    ])
    
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    
    # Column transformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='passthrough'  # Keep other columns
    )
    
    return preprocessor

# --- Feature engineering pipeline
def create_feature_pipeline(categorical_cols=['card4', 'P_emaildomain', 'R_emaildomain']):
    pipeline = Pipeline([
        ('time_features', TimeFeatureExtractor()),                 # 🕰️ Extract time features
        ('user_id', UserIDCreator(version=1)),                     # 🪪 Create user ID
        ('freq_encoding', FrequencyEncoder(columns=categorical_cols)),         # 🧮 Frequency encoding
    ])
    return pipeline

# --- Final pipeline with target-dependent features (for training only)
def create_training_pipeline(categorical_cols=['card4', 'P_emaildomain', 'R_emaildomain']):
    # This pipeline includes encoders that need the target variable
    pipeline = Pipeline([
        ('woe_encoding', WOEEncoder(columns=categorical_cols, target_col='isFraud')),     # 📈 WOE encoding
        ('target_encoding', TargetEncoder(columns=categorical_cols, target_col='isFraud')) # 🎯 Target encoding
    ])
    return pipeline

# --- Timestamp train/test splitter
def timestamp_train_test_split(X, y, split_day_threshold=250):
    train_idx = X['Transaction_day'] <= split_day_threshold
    test_idx = X['Transaction_day'] > split_day_threshold
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

# ========== MAIN TRAINING SCRIPT ==========

def main():
    # ⚡ Load your data
    df = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_transaction.csv')
    y = df['isFraud']
    
    # 🔍 Select a subset of features for demonstration if needed
    # df = df[selected_features + ['TransactionDT', 'isFraud']]
    
    # 🏗️ Apply feature engineering
    feature_pipe = create_feature_pipeline()
    X_featurized = feature_pipe.fit_transform(df)
    
    # ⏳ Train-test split by timestamp
    X_train, X_test, y_train, y_test = timestamp_train_test_split(X_featurized, y)
    
    # 🎭 Apply target-dependent encodings on training data
    training_pipe = create_training_pipeline()
    X_train_encoded = training_pipe.fit_transform(X_train, y_train)
    X_test_encoded = training_pipe.transform(X_test)
    
    # 🧰 Preprocess the data
    preprocessor = create_preprocessing_pipeline()
    X_train_preprocessed = preprocessor.fit_transform(X_train_encoded)
    X_test_preprocessed = preprocessor.transform(X_test_encoded)
    
    # 🌲 Random Forest
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    rf.fit(X_train_preprocessed, y_train)
    rf_preds = rf.predict_proba(X_test_preprocessed)[:, 1]
    rf_auc = roc_auc_score(y_test, rf_preds)
    
    
    # 📢 Results
    print(f"Random Forest AUC: {rf_auc:.5f}")

if __name__ == "__main__":
    main()

ValueError: A given column is not a column of the dataframe